# Modelo Predictivo de Puntos — LigaPro Ecuador
### Version Google Colab (datos reales 2019-2026)

**Que predice este modelo?**

Dado un equipo **ANTES de que empiece la temporada**, estima cuantos **puntos alcanzara al final del campeonato** usando solo variables conocidas en pre-temporada:

| Bloque | Variables |
|--------|-----------|
| Economicas | Presupuesto, salarios, inversion en refuerzos |
| Plantilla | Edad promedio, jugadores extranjeros |
| Cuerpo tecnico | Antiguedad del DT |
| Contexto | Capacidad estadio, participacion internacional |
| Temporada anterior | Posicion, goles a favor/contra, xG, posesion |

**Que NO incluye:** Goles, xG o posesion de la temporada en curso (esos se conocen despues de jugar — incluirlos seria trampa estadistica).

---
**Pasos antes de correr:**
1. Sube la carpeta `ligapro Seria A/` a Google Drive.
2. Incluye adentro el archivo `data/DATOS FINANZAS - LIGAPRO 2019-2026.xlsx`.
3. Abre este `.ipynb` -> clic derecho -> **Abrir con -> Google Colaboratory**.
4. Ejecuta las celdas en orden.

## PASO 1 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ajusta esta ruta si guardaste la carpeta con otro nombre/ubicacion en Drive
PROJECT_PATH = '/content/drive/MyDrive/ligapro Seria A'

import os, sys
os.chdir(PROJECT_PATH)
sys.path.insert(0, f'{PROJECT_PATH}/scripts')
print('Directorio:', os.getcwd())

## PASO 2 — Instalar dependencias

In [ ]:
!pip install --quiet xgboost joblib openpyxl
print('OK')

## PASO 3 — Ingesta de datos reales desde Excel

Lee `data/DATOS FINANZAS - LIGAPRO 2019-2026.xlsx` y genera los 8 CSV en `data/phase1_raw_snapshot/`. Omite esta celda si ya regeneraste los CSV antes.

In [ ]:
import subprocess
r = subprocess.run(
    ['python', 'scripts/cargar_datos_reales.py',
     '--excel', 'data/DATOS FINANZAS - LIGAPRO 2019-2026.xlsx'],
    capture_output=True, text=True
)
print(r.stdout)
if r.returncode != 0:
    print('ERROR:', r.stderr)

## PASO 4 — Consolidar dataset con variables rezagadas

Une los 8 CSV crudos y genera las variables de la temporada anterior (`goles_favor_ant`, `xg_ant`, etc.).

In [ ]:
r = subprocess.run(['python', 'scripts/consolidar_dataset.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('ERROR:', r.stderr)

## PASO 5 — Imports y carga del dataset

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
from preprocesamiento import FEATURES, TARGET, construir_xy, particion_temporal
from consolidar_dataset import consolidar
print(f'TARGET  : {TARGET}')
print(f'FEATURES ({len(FEATURES)}): {FEATURES}')

In [ ]:
df = consolidar()
print(f'Dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
df.head()

## PASO 6 — EDA

### 6.1. Presupuesto vs puntos

In [ ]:
fig, ax = plt.subplots()
for temp, g in df.groupby('temporada'):
    ax.scatter(g['presupuesto_musd'], g['puntos_temporada'], label=str(temp), alpha=0.8, s=60)
ax.set_xlabel('Presupuesto (MUSD)')
ax.set_ylabel('Puntos al final de temporada')
ax.set_title('Presupuesto vs puntos — LigaPro 2019-2026')
ax.legend(title='Temporada')
plt.tight_layout(); plt.show()

### 6.2. Puntos promedio por club

In [ ]:
pts = df.groupby('equipo')['puntos_temporada'].mean().sort_values(ascending=True)
pts.plot(kind='barh', color='#2E4F85', figsize=(9,7))
plt.xlabel('Puntos promedio (8 temporadas)')
plt.title('Rendimiento promedio — LigaPro 2019-2026')
plt.tight_layout(); plt.show()

### 6.3. Correlacion con el target

In [ ]:
corr = df[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
colors = ['#2E4F85' if v >= 0 else '#C0392B' for v in corr]
corr.plot(kind='barh', color=colors, figsize=(9,6))
plt.axvline(0, color='k', lw=0.8)
plt.title('Correlacion con puntos_temporada (azul = positiva, rojo = negativa)')
plt.tight_layout(); plt.show()
print('Top 3 variables mas correlacionadas:')
print(corr.abs().sort_values(ascending=False).head(3).to_string())

## PASO 7 — Particion temporal y entrenamiento

In [ ]:
TEMPORADAS_TRAIN = [2019, 2020, 2021, 2022, 2023]
TEMPORADAS_VAL   = [2024]
TEMPORADAS_TEST  = [2025]
train, val, test = particion_temporal(df, TEMPORADAS_TRAIN, TEMPORADAS_VAL, TEMPORADAS_TEST)
train = train.dropna(subset=[TARGET] + FEATURES)
val   = val.dropna(subset=[TARGET] + FEATURES)
test  = test.dropna(subset=[TARGET] + FEATURES)
X_train, y_train, _         = construir_xy(train)
X_val,   y_val,   meta_val  = construir_xy(val)
X_test,  y_test,  meta_test = construir_xy(test)
print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

In [ ]:
def spearman(a, b):
    return float(np.corrcoef(pd.Series(a).rank(), pd.Series(b).rank())[0, 1])

def evaluar(yt, yp):
    return {'MAE': round(float(mean_absolute_error(yt, yp)), 2),
            'RMSE': round(float(np.sqrt(mean_squared_error(yt, yp))), 2),
            'R2': round(float(r2_score(yt, yp)), 3),
            'Spearman': round(float(spearman(yt, yp)), 3)}

try:
    from xgboost import XGBRegressor
    xgb = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                       subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1, verbosity=0)
    xgb_name = 'XGBoost'
except Exception:
    xgb = GradientBoostingRegressor(n_estimators=400, learning_rate=0.05, max_depth=3, random_state=42)
    xgb_name = 'GradientBoosting'

modelos = {
    'Ridge':        Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=1.0, random_state=42))]),
    'RandomForest': Pipeline([('s', StandardScaler(with_mean=False)),
                              ('m', RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=42, n_jobs=-1))]),
    xgb_name:       Pipeline([('s', StandardScaler(with_mean=False)), ('m', xgb)]),
}
resultados, predicciones = {}, {}
for nombre, pipe in modelos.items():
    pipe.fit(X_train, y_train)
    pv = pipe.predict(X_val)
    pt = pipe.predict(X_test) if len(X_test) else np.array([])
    resultados[nombre]   = {'val': evaluar(y_val, pv)}
    if len(pt):
        resultados[nombre]['test'] = evaluar(y_test, pt)
    predicciones[nombre] = {'val': pv, 'test': pt}
    print(f'[{nombre}]  Val -> {resultados[nombre]["val"]}')

In [ ]:
resumen = pd.DataFrame({k: v['val'] for k, v in resultados.items()}).T
display(resumen.style
    .highlight_min(color='#c6efce', subset=['MAE', 'RMSE'])
    .highlight_max(color='#c6efce', subset=['R2', 'Spearman'])
    .set_caption('Validacion 2024 — MAE en PUNTOS, verde = mejor'))
mejor_nombre = min(resultados, key=lambda k: resultados[k]['val']['MAE'])
print(f'Mejor modelo: {mejor_nombre}')
print(f'  MAE val  = {resultados[mejor_nombre]["val"]["MAE"]} pts')
if 'test' in resultados[mejor_nombre]:
    print(f'  MAE test = {resultados[mejor_nombre]["test"]["MAE"]} pts')

## PASO 8 — Importancia de variables

In [ ]:
m_final = modelos[mejor_nombre].named_steps['m']
if hasattr(m_final, 'feature_importances_'):
    imp = pd.Series(m_final.feature_importances_, index=FEATURES).sort_values(ascending=True)
    titulo = f'Que variable explica mas los puntos? ({mejor_nombre})'
elif hasattr(m_final, 'coef_'):
    imp = pd.Series(m_final.coef_, index=FEATURES).sort_values(key=lambda s: s.abs(), ascending=True)
    titulo = f'Coeficientes estandarizados ({mejor_nombre})'
else:
    imp = None
    titulo = None
if imp is not None:
    imp.plot(kind='barh', color='#1F3B6B', figsize=(9, 6))
    plt.title(titulo)
    plt.xlabel('Peso / coeficiente')
    plt.tight_layout(); plt.show()
    print('Top 5:')
    print(imp.reindex(imp.abs().sort_values(ascending=False).index).head(5).to_string())

## PASO 9 — Predicciones temporada de prueba (2025)

Cuantos puntos predice el modelo para cada equipo **antes de jugar**?

In [ ]:
pred_test = predicciones[mejor_nombre]['test']
if len(pred_test):
    tabla = meta_test.copy()
    tabla['puntos_reales']    = y_test.to_numpy()
    tabla['puntos_predichos'] = np.round(pred_test, 1)
    tabla['error_pts']        = np.abs(tabla['puntos_reales'] - tabla['puntos_predichos']).round(1)
    tabla = tabla.sort_values('puntos_predichos', ascending=False).reset_index(drop=True)
    tabla.index += 1
    display(tabla.style
        .background_gradient(cmap='RdYlGn', subset=['puntos_predichos'])
        .background_gradient(cmap='RdYlGn_r', subset=['error_pts'])
        .set_caption(f'Prediccion pre-temporada 2025 — {mejor_nombre}'))
    print(f'MAE: {tabla["error_pts"].mean():.1f} puntos')
else:
    tabla = meta_val.copy()
    tabla['puntos_reales']    = y_val.to_numpy()
    tabla['puntos_predichos'] = np.round(predicciones[mejor_nombre]['val'], 1)
    print('Sin test disponible; se muestra la validacion 2024.')
    display(tabla.head(16))

In [ ]:
fig, ax = plt.subplots()
ax.scatter(tabla['puntos_reales'], tabla['puntos_predichos'], s=80, color='#2E4F85', zorder=3)
for _, row in tabla.iterrows():
    ax.annotate(row['equipo'].split()[0],
                (row['puntos_reales'], row['puntos_predichos']),
                fontsize=7, alpha=0.85, xytext=(3, 2), textcoords='offset points')
lims = [10, 90]
ax.plot(lims, lims, 'r--', alpha=0.5, label='Prediccion perfecta')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Puntos reales'); ax.set_ylabel('Puntos predichos')
ax.set_title(f'Real vs predicho — {mejor_nombre}')
ax.legend()
plt.tight_layout(); plt.show()

## PASO 10 — Guardar modelos en Drive

In [ ]:
os.makedirs('models', exist_ok=True)
mejor = modelos[mejor_nombre]
joblib.dump(mejor, 'models/mejor_modelo.joblib')
for nombre, pipe in modelos.items():
    joblib.dump(pipe, f'models/{nombre.lower()}.joblib')
with open('models/model_card.json', 'w', encoding='utf-8') as f:
    json.dump({'modelos': resultados,
               'mejor_modelo': mejor_nombre,
               'features': FEATURES,
               'target': TARGET,
               'particion': {'train': TEMPORADAS_TRAIN,
                             'val': TEMPORADAS_VAL,
                             'test': TEMPORADAS_TEST}}, f, indent=2)
print('Guardado en Drive: models/')

## Que esperar de este modelo?

Con el dataset real 2019-2026 (128 filas, 14 variables pre-temporada):

| Metrica | Ridge (ganador) | Random Forest | Gradient Boosting |
|---------|-----------------|---------------|-------------------|
| MAE (val 2024) | 6.52 pts | 9.32 | 9.38 |
| R2 (val 2024) | 0.63 | 0.27 | 0.30 |
| Spearman | 0.75 | 0.46 | 0.35 |

**En la practica:**
- El modelo acierta el cuadrante (campeon / zona media / descenso) en la mayoria de equipos.
- Un error de ~6 puntos equivale a 2 partidos — razonable dada la varianza natural del futbol.
- Los equipos de presupuesto medio son los mas dificiles de predecir.

**Para actualizar con nuevos datos:**
- Reemplaza el archivo `data/DATOS FINANZAS - LIGAPRO 2019-2026.xlsx` y vuelve a correr desde el **PASO 3**.
- Ajusta `TEMPORADAS_TRAIN / VAL / TEST` segun el historial disponible.